In [ ]:
import pandas as pd
from sqlalchemy import text
from connection import connect

# Conexion
co_oltp, etl_conn, etl_conn_or = connect()

# Extraccion de datos del OLTP
query_reseller_sales = text("""
SELECT 
    sod.sales_order_id,
    soh.sales_order_number,
    sod.sales_order_detail_id,
    ROW_NUMBER() OVER (PARTITION BY soh.sales_order_number ORDER BY sod.sales_order_detail_id) AS sales_order_line_number,
    sod.order_qty,
    sod.unit_price,
    (sod.unit_price * sod.order_qty) AS extended_amount,
    sod.unit_price_discount,
    (sod.unit_price * sod.unit_price_discount * sod.order_qty) AS discount_amount,
    p.product_number,
    p.standard_cost AS product_standard_cost,
    (sod.order_qty * p.standard_cost) AS total_product_cost,
    (sod.unit_price * (1 - sod.unit_price_discount) * sod.order_qty) AS sales_amount,
    soh.tax_amt,
    soh.freight,
    soh.order_date,
    soh.due_date,
    soh.ship_date,
    soh.territory_id,
    soh.sales_person_id,
    soh.currency_rate_id,
    soh.purchase_order_number AS customer_po_number,
    soh.revision_number,
    soh.carrier_tracking_number,
    r.business_entity_id AS reseller_id
FROM sales.sales_order_header AS soh
JOIN sales.sales_order_detail AS sod
    ON soh.sales_order_id = sod.sales_order_id
JOIN production.product AS p
    ON sod.product_id = p.product_id
JOIN sales.customer AS c
    ON soh.customer_id = c.customer_id
JOIN sales.store AS r
    ON c.store_id = r.business_entity_id
WHERE soh.online_order_flag = false
""")

fact_reseller_sales = pd.read_sql(query_reseller_sales, co_oltp)
print(f"Registros extraidos: {len(fact_reseller_sales)}")
print(fact_reseller_sales.head(3))

# Dimensiones necesarias

## a. DimProduct
dim_product = pd.read_sql_table('dim_product', etl_conn)
fact_reseller_sales = fact_reseller_sales.merge(
    dim_product[['product_key', 'product_alternate_key']],
    left_on='product_number',
    right_on='product_alternate_key',
    how='left'
).drop(['product_number', 'product_alternate_key'], axis=1)

## b. DimReseller
dim_reseller = pd.read_sql_table('dim_reseller', etl_conn)
fact_reseller_sales = fact_reseller_sales.merge(
    dim_reseller[['reseller_key', 'reseller_alternate_key']],
    left_on='reseller_id',
    right_on='reseller_alternate_key',
    how='left'
).drop(['reseller_id', 'reseller_alternate_key'], axis=1)

## c. DimEmployee (vendedor asignado)
dim_employee = pd.read_sql_table('dim_employee', etl_conn)
fact_reseller_sales = fact_reseller_sales.merge(
    dim_employee[['employee_key', 'employee_alternate_key']],
    left_on='sales_person_id',
    right_on='employee_alternate_key',
    how='left'
).drop(['sales_person_id', 'employee_alternate_key'], axis=1)

## d. DimPromotion
t_special_offer = pd.read_sql("""
    SELECT p.product_number, sod.special_offer_id
    FROM sales.sales_order_detail AS sod
    JOIN production.product AS p ON sod.product_id = p.product_id
""", co_oltp)

dim_promotion = pd.read_sql_table('dim_promotion', etl_conn)
fact_reseller_sales = fact_reseller_sales.merge(
    t_special_offer,
    left_on='sales_order_detail_id',
    right_index=True,
    how='left'
)

fact_reseller_sales = fact_reseller_sales.merge(
    dim_promotion[['promotion_key', 'promotion_alternate_key']],
    left_on='special_offer_id',
    right_on='promotion_alternate_key',
    how='left'
).drop(['special_offer_id', 'promotion_alternate_key'], axis=1)

## e. DimCurrency
dim_currency = pd.read_sql_table('dim_currency', etl_conn)
t_currency = pd.read_sql("SELECT currency_rate_id, to_currency_code FROM sales.currency_rate", co_oltp)

fact_reseller_sales = fact_reseller_sales.merge(
    t_currency,
    on='currency_rate_id',
    how='left'
)

fact_reseller_sales = fact_reseller_sales.merge(
    dim_currency[['currency_key', 'currency_alternate_key']],
    left_on='to_currency_code',
    right_on='currency_alternate_key',
    how='left'
).drop(['currency_rate_id', 'currency_alternate_key', 'to_currency_code'], axis=1)

## f. DimSalesTerritory
dim_sales_territory = pd.read_sql_table('dim_sales_territory', etl_conn)
fact_reseller_sales = fact_reseller_sales.merge(
    dim_sales_territory[['sales_territory_key', 'sales_territory_alternate_key']],
    left_on='territory_id',
    right_on='sales_territory_alternate_key',
    how='left'
).drop(['territory_id', 'sales_territory_alternate_key'], axis=1)

# Claves de fechas (YYYYMMDD)
fact_reseller_sales['order_date_key'] = fact_reseller_sales['order_date'].dt.strftime('%Y%m%d').astype(int)
fact_reseller_sales['due_date_key'] = fact_reseller_sales['due_date'].dt.strftime('%Y%m%d').astype(int)
fact_reseller_sales['ship_date_key'] = fact_reseller_sales['ship_date'].dt.strftime('%Y%m%d').astype(int)

# Seleccion de columnas para la tabla FactResellerSales
final_columns = [
    'product_key',
    'order_date_key',
    'due_date_key',
    'ship_date_key',
    'reseller_key',
    'employee_key',
    'promotion_key',
    'currency_key',
    'sales_territory_key',
    'sales_order_number',
    'sales_order_line_number',
    'revision_number',
    'order_qty',
    'unit_price',
    'extended_amount',
    'unit_price_discount',
    'discount_amount',
    'product_standard_cost',
    'total_product_cost',
    'sales_amount',
    'tax_amt',
    'freight',
    'carrier_tracking_number',
    'customer_po_number',
    'order_date',
    'due_date',
    'ship_date'
]

df_to_load = fact_reseller_sales[final_columns]
print(f"Columnas finales: {df_to_load.columns.tolist()}")
print(df_to_load.head(3))

# Carga al DW
df_to_load.to_sql(
    'fact_reseller_sales',
    etl_conn,
    if_exists='append',
    index=False
)

print("Carga finalizada en FactResellerSales")
